In [1]:
import os, glob, cv2, pydicom, numpy as np, pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

DATA = os.path.dirname(glob.glob("/kaggle/input/**/stage_2_train_labels.csv", recursive=True)[0])
labels = pd.read_csv(f"{DATA}/stage_2_train_labels.csv")
info = pd.read_csv(f"{DATA}/stage_2_detailed_class_info.csv").drop_duplicates("patientId")

def read_meta(pid):
    d = pydicom.dcmread(f"{DATA}/stage_2_train_images/{pid}.dcm", stop_before_pixels=True)
    return {"patientId": pid, "age": int(d.PatientAge), "sex": d.PatientSex,
            "view": d.ViewPosition, "photometric": d.PhotometricInterpretation}

with ThreadPoolExecutor(8) as ex:
    rows = list(tqdm(ex.map(read_meta, info["patientId"]), total=len(info)))
meta = pd.DataFrame(rows).merge(info, on="patientId")
print(meta["class"].value_counts())

  0%|          | 0/26684 [00:00<?, ?it/s]

class
No Lung Opacity / Not Normal    11821
Normal                           8851
Lung Opacity                     6012
Name: count, dtype: int64


In [2]:
SIZE = 256
meta = meta.reset_index(drop=True)
meta["idx"] = meta.index            # row number inside the image array

imgs = np.lib.format.open_memmap("/kaggle/working/images_256.npy", mode="w+",
                                 dtype=np.uint8, shape=(len(meta), SIZE, SIZE))

def convert(i):
    arr = pydicom.dcmread(f"{DATA}/stage_2_train_images/{meta.at[i, 'patientId']}.dcm").pixel_array
    assert arr.dtype == np.uint8, arr.dtype
    imgs[i] = cv2.resize(arr, (SIZE, SIZE), interpolation=cv2.INTER_AREA)

with ThreadPoolExecutor(4) as ex:
    list(tqdm(ex.map(convert, range(len(meta))), total=len(meta)))
imgs.flush()
print("Saved:", imgs.shape)

label_map = {"Normal": 0, "No Lung Opacity / Not Normal": 1, "Lung Opacity": 2}
meta["label"] = meta["class"].map(label_map)

train, temp = train_test_split(meta, test_size=0.30, stratify=meta["label"], random_state=42)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=42)

for name, df in [("train", train), ("val", val), ("test", test)]:
    df.to_csv(f"/kaggle/working/{name}.csv", index=False)
    print(name, len(df), df["label"].value_counts(normalize=True).round(2).to_dict())

labels.to_csv("/kaggle/working/boxes.csv", index=False)

  0%|          | 0/26684 [00:00<?, ?it/s]

Saved: (26684, 256, 256)
train 18678 {1: 0.44, 0: 0.33, 2: 0.23}
val 4003 {1: 0.44, 0: 0.33, 2: 0.23}
test 4003 {1: 0.44, 0: 0.33, 2: 0.23}
